# Hybrid Retrieval for Retrieval-Augmented Generation (RAG)

## Overview

Traditional Retrieval-Augmented Generation (RAG) systems typically rely on **dense retrieval**, where documents are embedded into high-dimensional vectors and retrieved based on semantic similarity.

Although dense retrieval is effective for understanding meaning, it may fail when the query depends on exact keywords, abbreviations, formulas, or uncommon terminology.

Hybrid Retrieval addresses this limitation by combining:

- **Dense Retrieval** (semantic search using vector embeddings)
- **Sparse Retrieval** (lexical search using BM25)

The results from both retrieval methods are merged to produce a richer and more reliable context before generating a response.

---

## Objectives

In this notebook, we will:

- Load research papers
- Split documents into semantic chunks
- Build a Dense Retriever using ChromaDB
- Build a Sparse Retriever using BM25
- Combine both retrieval methods
- Compare Dense, Sparse, and Hybrid Retrieval
- Evaluate retrieval quality

---

## Hybrid Retrieval Pipeline

```

                    User Query
                         │
         ┌───────────────┴───────────────┐
         │                               │
         ▼                               ▼
 Dense Retrieval                  Sparse Retrieval
 (Embeddings)                          (BM25)
         │                               │
         └───────────────┬───────────────┘
                         ▼
                  Merge Results
                         ▼
                   Remove Duplicates
                         ▼
                    Retrieved Context
                         ▼
                         LLM

```

In [32]:
# ==========================================================
# Import Required Libraries
# ==========================================================

import os

from dotenv import load_dotenv

import chromadb

from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
)

from llama_index.core.node_parser import SentenceSplitter

from llama_index.embeddings.openai import OpenAIEmbedding

from llama_index.vector_stores.chroma import ChromaVectorStore

from llama_index.llms.openai import OpenAI

from openai import OpenAI as OpenRouterClient

from llama_index.llms.openai_like import OpenAILike

In [33]:
# ==========================================================
# Load Environment Variables
# ==========================================================

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found.")

In [34]:
# ==========================================================
# Project Configuration
# ==========================================================

DATA_DIRECTORY = "../data"

CHROMA_DB_PATH = "../storage/chroma_db"

COLLECTION_NAME = "hybrid_research"

EMBEDDING_MODEL = "text-embedding-3-large"

LLM_MODEL = "openrouter/free"

CHUNK_SIZE = 512

CHUNK_OVERLAP = 100

TOP_K = 5

# Understanding Sparse Retrieval (BM25)

Before implementing Hybrid Retrieval, it is important to understand why dense retrieval alone is not always sufficient.

## Dense Retrieval

Dense retrieval converts both documents and queries into dense vector embeddings. These embeddings capture semantic meaning, allowing the retriever to find relevant documents even when the wording differs.

Example:

Query:
> "How does the system learn from data?"

Retrieved chunk:
> "Machine learning algorithms improve performance by learning patterns."

Although the exact words differ, the semantic meaning is similar.

---

## Limitations of Dense Retrieval

Dense retrieval may struggle with:

- Acronyms (CNN, RNN, LSTM)
- Variable names
- Mathematical formulas
- Error codes
- Exact terminology
- Rare technical keywords

For example:

Query:
> "CNN"

A dense retriever may incorrectly retrieve chunks discussing neural networks in general instead of **Convolutional Neural Networks**.

---

## Sparse Retrieval (BM25)

BM25 is a lexical retrieval algorithm.

Instead of understanding semantic meaning, BM25 searches for exact keyword matches and ranks documents based on:

- Term Frequency (TF)
- Inverse Document Frequency (IDF)
- Document Length Normalization

Because of this, BM25 performs exceptionally well when exact keywords matter.

---

## Why Hybrid Retrieval?

Dense retrieval captures semantic similarity.

Sparse retrieval captures lexical similarity.

By combining both approaches, Hybrid Retrieval improves recall and robustness across a wider range of queries.

```
                    User Query
                         │
         ┌───────────────┴───────────────┐
         │                               │
         ▼                               ▼
 Dense Retrieval                  BM25 Retrieval
 (Semantic Search)             (Keyword Search)
         │                               │
         └───────────────┬───────────────┘
                         ▼
                   Merge Results
                         ▼
                      Final Context
```

This combination is widely adopted in modern Retrieval-Augmented Generation (RAG) systems.

In [35]:
# ==========================================================
# Import BM25 Retriever
# ==========================================================

from llama_index.retrievers.bm25 import BM25Retriever

In [36]:
# ==========================================================
# Load Research Papers
# ==========================================================

documents = SimpleDirectoryReader(
    input_dir=DATA_DIRECTORY
).load_data()

print(f"Loaded {len(documents)} documents.")

2026-07-30 01:18:35,159 - WARNING - Ignoring wrong pointing object 6 0 (offset 0)
2026-07-30 01:18:35,161 - WARNING - Ignoring wrong pointing object 8 0 (offset 0)
2026-07-30 01:18:35,162 - WARNING - Ignoring wrong pointing object 10 0 (offset 0)
2026-07-30 01:18:35,164 - WARNING - Ignoring wrong pointing object 12 0 (offset 0)
2026-07-30 01:18:35,165 - WARNING - Ignoring wrong pointing object 14 0 (offset 0)
2026-07-30 01:18:35,166 - WARNING - Ignoring wrong pointing object 16 0 (offset 0)
2026-07-30 01:18:35,168 - WARNING - Ignoring wrong pointing object 18 0 (offset 0)
2026-07-30 01:18:35,169 - WARNING - Ignoring wrong pointing object 20 0 (offset 0)
2026-07-30 01:18:35,170 - WARNING - Ignoring wrong pointing object 22 0 (offset 0)
2026-07-30 01:18:35,170 - WARNING - Ignoring wrong pointing object 24 0 (offset 0)
2026-07-30 01:18:35,172 - WARNING - Ignoring wrong pointing object 36 0 (offset 0)
2026-07-30 01:18:35,173 - WARNING - Ignoring wrong pointing object 48 0 (offset 0)


Loaded 42 documents.


In [37]:
# ==========================================================
# Split Documents into Nodes
# ==========================================================

splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

nodes = splitter.get_nodes_from_documents(documents)

print(f"Created {len(nodes)} chunks.")

Created 124 chunks.


In [38]:
# ==========================================================
# Create Dense Retriever
# ==========================================================

embed_model = OpenAIEmbedding(
    api_key=OPENROUTER_API_KEY,
    api_base="https://openrouter.ai/api/v1",
    model=EMBEDDING_MODEL,
)

db = chromadb.PersistentClient(
    path=CHROMA_DB_PATH
)

collection = db.get_or_create_collection(
    COLLECTION_NAME
)

vector_store = ChromaVectorStore(
    chroma_collection=collection
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    embed_model=embed_model,
)

dense_retriever = index.as_retriever(
    similarity_top_k=TOP_K
)

print("Dense Retriever Ready")

2026-07-30 01:18:40,385 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
2026-07-30 01:18:41,756 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"


Dense Retriever Ready


In [39]:
# ==========================================================
# Create BM25 Retriever
# ==========================================================

bm25_retriever = BM25Retriever.from_defaults(
    nodes=nodes,
    similarity_top_k=TOP_K,
)

print("BM25 Retriever Ready")

2026-07-30 01:18:42,311 - DEBUG - Building index from IDs objects


BM25 Retriever Ready


# Building a Custom Hybrid Retriever

Instead of using only Dense Retrieval or BM25 independently, we create a custom
Hybrid Retriever that combines the strengths of both approaches.

### Retrieval Process

1. Retrieve relevant chunks using Dense Retrieval.
2. Retrieve relevant chunks using BM25.
3. Merge the retrieved chunks.
4. Remove duplicate chunks.
5. Return the combined results.

This modular design allows us to easily experiment with different retrieval
strategies, weighting schemes, and rerankers in future notebooks.

In [40]:
# ==========================================================
# Custom Hybrid Retriever
# ==========================================================

from llama_index.core.retrievers import BaseRetriever


class HybridRetriever(BaseRetriever):
    """
    Hybrid Retriever combining Dense Retrieval and BM25 Retrieval.
    """

    def __init__(
        self,
        dense_retriever,
        bm25_retriever,
    ):
        super().__init__()

        self.dense_retriever = dense_retriever
        self.bm25_retriever = bm25_retriever

    def _retrieve(self, query_bundle):

        dense_results = self.dense_retriever.retrieve(query_bundle)

        bm25_results = self.bm25_retriever.retrieve(query_bundle)

        combined = {}

        # Add dense results
        for node in dense_results:
            combined[node.node.node_id] = node

        # Add BM25 results
        for node in bm25_results:
            if node.node.node_id not in combined:
                combined[node.node.node_id] = node

        return list(combined.values())

In [41]:
# ==========================================================
# Initialize Hybrid Retriever
# ==========================================================

hybrid_retriever = HybridRetriever(
    dense_retriever=dense_retriever,
    bm25_retriever=bm25_retriever,
)

print("Hybrid Retriever Ready")

Hybrid Retriever Ready


# Configure the Large Language Model (LLM)

The retrieved context is passed to a Large Language Model (LLM) to generate
a final response.

In this notebook, we use OpenRouter as the inference provider while
maintaining compatibility with the LlamaIndex framework.

In [42]:
# ==========================================================
# Initialize LLM
# ==========================================================

llm = OpenAILike(
    model=LLM_MODEL,
    api_key=OPENROUTER_API_KEY,
    api_base="https://openrouter.ai/api/v1",
    is_chat_model=True,
    timeout=300,
)

In [43]:
# ==========================================================
# Create Hybrid Query Engine
# ==========================================================

from llama_index.core.query_engine import RetrieverQueryEngine

query_engine = RetrieverQueryEngine.from_args(
    retriever=hybrid_retriever,
    llm=llm,
)

print("Hybrid Query Engine Ready")

Hybrid Query Engine Ready


In [44]:
# ==========================================================
# Test Hybrid Retrieval
# ==========================================================

query = "What role does machine learning play in the system?"

response = query_engine.query(query)

print(response)

2026-07-30 01:18:43,249 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
2026-07-30 01:18:44,170 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-30 01:18:48,879 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Machine learning is employed to scrutinise each e‑prescription for errors and safety risks.  
Before a prescription is finalized, ML models analyse the entered data to flag missing fields, misplaced information, incorrect dosages, and potential drug‑drug interactions. The system then generates alerts that are presented to the prescriber and pharmacist, allowing them to correct issues immediately. Because patients do not interact directly with the alert‑generation module, the ML component operates behind the scenes to enhance the overall safety, security, and reliability of the e‑prescription workflow.
